# 21 · Localização do 156 e histórico por hexágono (S4, G-I1, C1)

Leva cada demanda à grade H3 e registra **método e precisão**:

1. coordenada válida dentro do domínio (IBGE ∪ bairros) → hexágono do ponto;
2. sem coordenada → rua + bairro pela relação oficial (nome não ambíguo). A demanda é **repartida** pelos hexágonos que a rua cruza no bairro (pesos somam 1). Nenhum ponto fictício é criado;
3. o resto fica sem lugar, com o motivo.

A caixa de coordenadas da Silver é só um primeiro filtro: a validação territorial é o contorno.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from wastestack.arquivos import ler
from wastestack.config import SILVER, GOLD

In [2]:
from wastestack.gold import localizar_156, historico_h3_mes
resumo = localizar_156()
resumo

{
 "demandas": 230491,
 "localizadas": 214813,
 "pct_localizadas": 93.2,
 "recorte": 61790,
 "recorte_pct_localizado": 94.2,
 "recorte_por_metodo_pct": {
  "logradouro+bairro": 55.9,
  "coordenada": 38.3,
  "sem localização": 5.8
 },
 "coordenadas_fora_do_dominio_ou_da_grade": 355,
 "motivos_sem_localizacao_recorte": {
  "rua não encontrada na Área urbana": 2489,
  "rua existe, mas não neste bairro (relação oficial)": 1075,
  "nome de rua ambíguo no bairro": 47,
  "bairro não informado ou não reconhecido": 2
 },
 "precisao_rua_mediana_m": 428.3318135928732,
 "hexagonos_por_demanda_localizada_por_rua_mediana": 2.0,
 "cobertura_por_quartil_v0_do_bairro": {
  "Q1 (menor V0)": {
   "pct_localizada": 92.4,
   "pct_por_coordenada": 35.7
  },
  "Q2": {
   "pct_localizada": 94.6,
   "pct_por_coordenada": 38.1
  },
  "Q3": {
   "pct_localizada": 95.7,
   "pct_por_coordenada": 42.1
  },
  "Q4 (maior V0)": {
   "pct_localizada": 94.2,
   "pct_por_coordenada": 36.7
  }
 },
 "limite": "localização 

In [3]:
historico = historico_h3_mes()
historico

{
 "linhas": 47489,
 "hexagonos_com_historico": 1327,
 "chamados_somados": 54889.0,
 "meses": 25
}


In [4]:
assert resumo["recorte"] == 61790
print("Recorte localizado: %.1f%% (por coordenada %.1f%%; por rua+bairro %.1f%%)" % (resumo["recorte_pct_localizado"], resumo["recorte_por_metodo_pct"]["coordenada"], resumo["recorte_por_metodo_pct"]["logradouro+bairro"]))
ler("cobertura_localizacao_156_bairro").nsmallest(8, "pct_localizada")[["bairro_chave", "demandas", "pct_localizada", "pct_por_coordenada", "quartil_v0"]]

Recorte localizado: 94.2% (por coordenada 38.3%; por rua+bairro 55.9%)
          bairro_chave  demandas  pct_localizada  pct_por_coordenada     quartil_v0
35         DOIS IRMAOS       213       64.788732           21.596244  Q1 (menor V0)
71     PONTO DE PARADA       154       77.922078           35.064935             Q3
59          MANGABEIRA       222       80.630631           30.180180  Q1 (menor V0)
89                TOTO        94       81.914894           28.723404             Q2
3        ALTO DO MANDU       156       82.051282           38.461538             Q3
11            BEBERIBE       615       85.040650           30.406504  Q1 (menor V0)
14  BOMBA DO HEMETERIO       393       85.496183           24.936387  Q1 (menor V0)
39          ESPINHEIRO       226       85.840708           33.628319  Q4 (maior V0)


## Limites e cuidados

- A localização por rua tem precisão igual ao comprimento da rua no bairro (mediana na saída acima). Não é o ponto do chamado.
- A cobertura por quartil de V0 do bairro mostra se a localização favorece áreas menos precárias.
- O histórico por hexágono soma **pesos** (frações de demanda) e usa só os 25 meses completos.